# Datamart root-cause checks (Feb 2026, ultra-light)

## 1) Parameters

In [ ]:
import re
from decimal import Decimal, InvalidOperation

import pandas as pd
from rail_connectors.connection import connect

report_month = '2026-02-01'
month_start = '2026-02-01'
month_end = '2026-02-28'

excel_path = '/home/jovyan/documents/Equaring/Data/02_Февраль_2026.xlsx'
excel_inn_col = 'ИНН'
excel_header = 0

mem_limit = '8g'

In [ ]:
def normalize_numeric_str(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace(' ', '').replace('\xa0', '')
    if s == '' or s.lower() in {'nan', 'none', 'null'}:
        return None
    s = s.replace(',', '.')
    if re.search(r'[eE][+-]?\d+$', s):
        try:
            s = format(Decimal(s), 'f')
        except InvalidOperation:
            return None
    s = re.sub(r'\.0+$', '', s)
    s = re.sub(r'\D', '', s)
    if not s:
        return None
    if len(s) == 9:
        s = '0' + s
    elif len(s) == 11:
        s = '0' + s
    return s


def normalize_contract(v):
    if pd.isna(v):
        return None
    s = str(v).strip().lower()
    s = re.sub(r'\s+', '', s)
    return s if s else None

## 2) Connect to Impala

In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()

## 3) Load Excel INN benchmark

In [ ]:
excel_raw = pd.read_excel(excel_path, header=excel_header)
excel_raw.columns = [str(c).strip() for c in excel_raw.columns]

if excel_inn_col not in excel_raw.columns:
    raise ValueError(f'Missing Excel INN column: {excel_inn_col}')

excel_df = excel_raw[[excel_inn_col]].copy()
excel_df.columns = ['inn_raw']
excel_df['inn'] = excel_df['inn_raw'].apply(normalize_numeric_str)
excel_df = excel_df[(excel_df['inn'].notna()) & (excel_df['inn'] != '')].copy()
excel_df = excel_df[['inn']].drop_duplicates().sort_values('inn').reset_index(drop=True)

excel_inn_set = set(excel_df['inn'].tolist())
len(excel_inn_set)

## 4) Build SA perimeter (agreements+companies)

In [ ]:
sql_agr = f"""
select distinct
  regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn,
  cast(a.abs_agr_id as string) as agr_id,
  cast(a.c_agr_number as string) as agreement_contract_number
from ods_alpha.scd1_agreements a
join ods_alpha.scd1_companies c
  on c.n_cmp = a.n_cmp_client
where upper(trim(cast(a.acq_class as string))) = 'SA'
  and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
  and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
  and coalesce(a.ods_deleted_flg, '0') <> '1'
  and coalesce(c.ods_deleted_flg, '0') <> '1'
  and c.c_inn is not null
"""

with imp:
    imp.execute(f"set MEM_LIMIT={mem_limit}")
    agr_df = imp.fetch(sql_agr)

agr_df['inn'] = agr_df['inn'].apply(normalize_numeric_str)
agr_df['agr_id'] = agr_df['agr_id'].astype(str)
agr_df.loc[agr_df['agr_id'].isin(['None', 'nan', '']), 'agr_id'] = None
agr_df['agreement_contract_number'] = agr_df['agreement_contract_number'].apply(normalize_contract)
agr_df = agr_df[(agr_df['inn'].notna()) & (agr_df['inn'] != '')].drop_duplicates().reset_index(drop=True)
agr_df['row_uid'] = agr_df.index.astype(str)

len(agr_df)

## 5) Enrich CDI_ID / CFT_ID

In [ ]:
inn_values = sorted(agr_df['inn'].dropna().unique().tolist())
inn_values_sql = ', '.join(["'" + str(x).replace("'", "''") + "'" for x in inn_values])

sql_inn_to_cdi = f"""
with ocrm as (
  select
    regexp_replace(trim(cast(se.x_inn as string)), '[^0-9]', '') as inn,
    cast(se.row_id as string) as row_id,
    row_number() over (
      partition by regexp_replace(trim(cast(se.x_inn as string)), '[^0-9]', '')
      order by cast(se.last_upd as timestamp) desc,
               cast(se.created as timestamp) desc,
               cast(se.row_id as string) desc
    ) as rn
  from ocrm_ul.s_org_ext se
  where se.x_removed_flg = 'N'
    and se.x_duplicate_flg = 'N'
    and regexp_replace(trim(cast(se.x_inn as string)), '[^0-9]', '') in ({inn_values_sql})
)
select
  o.inn,
  cast(e.party_id as string) as cdi_id
from ocrm o
left join cdiul.ext_id_org e
  on cast(e.cmo_ext_party_source_id as string) = o.row_id
 and upper(e.cmo_ext_source_system) like 'OCRM%'
where o.rn = 1
"""

with imp:
    imp.execute(f"set MEM_LIMIT={mem_limit}")
    inn_to_cdi_df = imp.fetch(sql_inn_to_cdi)

inn_to_cdi_df['inn'] = inn_to_cdi_df['inn'].apply(normalize_numeric_str)
inn_to_cdi_df['cdi_id'] = inn_to_cdi_df['cdi_id'].astype(str)
inn_to_cdi_df.loc[inn_to_cdi_df['cdi_id'].isin(['None', 'nan', '']), 'cdi_id'] = None
inn_to_cdi_df = inn_to_cdi_df.drop_duplicates()

cdi_values = sorted(inn_to_cdi_df['cdi_id'].dropna().unique().tolist())
if len(cdi_values):
    cdi_values_sql = ', '.join(["'" + str(x).replace("'", "''") + "'" for x in cdi_values])
    sql_cdi_to_cft = f"""
    select distinct
      cast(party_id as string) as cdi_id,
      cast(cmo_ext_party_source_id as string) as cft_id
    from cdiul.ext_id_org
    where upper(cmo_ext_source_system) like 'CFT%'
      and cast(party_id as string) in ({cdi_values_sql})
    """
    with imp:
        imp.execute(f"set MEM_LIMIT={mem_limit}")
        cdi_to_cft_df = imp.fetch(sql_cdi_to_cft)
else:
    cdi_to_cft_df = pd.DataFrame(columns=['cdi_id', 'cft_id'])

if len(cdi_to_cft_df):
    cdi_to_cft_df['cdi_id'] = cdi_to_cft_df['cdi_id'].astype(str)
    cdi_to_cft_df['cft_id'] = cdi_to_cft_df['cft_id'].astype(str)
    cdi_to_cft_df.loc[cdi_to_cft_df['cft_id'].isin(['None', 'nan', '']), 'cft_id'] = None
    cdi_to_cft_df = cdi_to_cft_df.drop_duplicates()

cdi_to_cft_one = cdi_to_cft_df.sort_values('cft_id').drop_duplicates('cdi_id') if len(cdi_to_cft_df) else cdi_to_cft_df
inn_cdi_cft_df = inn_to_cdi_df.merge(cdi_to_cft_one, on='cdi_id', how='left').drop_duplicates()

len(inn_cdi_cft_df)

## 6) Enrich R2 and contract number

In [ ]:
agr_id_values = sorted(agr_df['agr_id'].dropna().unique().tolist())
if len(agr_id_values):
    agr_id_values_sql = ', '.join(["'" + str(x).replace("'", "''") + "'" for x in agr_id_values])
    sql_r2_best = f"""
    with m as (
      select
        cast(id as string) as r2_id,
        cast(c_cl_org as string) as cft_id,
        cast(c_code_in_pr as string) as c_code_in_pr
      from ods.scd1_z_r2_ip_merchants
      where cast(id as string) in ({agr_id_values_sql})
    ),
    d as (
      select
        cast(id as string) as dog_id,
        cast(c_parent_id as string) as dog_parent_id,
        cast(c_code_in_pr as string) as dog_code_in_pr,
        cast(c_num as string) as dog_contract_number
      from ods.scd1_z_r2_ip_merch_dog
    ),
    candidate as (
      select m.r2_id, m.cft_id, d.dog_contract_number, 'join_by_parent_id' as join_type, 1 as rank, d.dog_id
      from m left join d on d.dog_parent_id = m.r2_id
      union all
      select m.r2_id, m.cft_id, d.dog_contract_number, 'join_by_id' as join_type, 2 as rank, d.dog_id
      from m left join d on d.dog_id = m.r2_id
      union all
      select m.r2_id, m.cft_id, d.dog_contract_number, 'join_by_code_in_pr' as join_type, 3 as rank, d.dog_id
      from m left join d on d.dog_code_in_pr is not null and m.c_code_in_pr is not null and d.dog_code_in_pr = m.c_code_in_pr
      union all
      select m.r2_id, m.cft_id, cast(null as string) as dog_contract_number, 'no_match' as join_type, 9 as rank, cast(null as string) as dog_id
      from m
    ),
    ranked as (
      select *, row_number() over (
        partition by r2_id
        order by rank, case when dog_contract_number is null then 1 else 0 end, dog_id
      ) as rn
      from candidate
    )
    select r2_id, cft_id, dog_contract_number, join_type
    from ranked
    where rn = 1
    """
    with imp:
        imp.execute(f"set MEM_LIMIT={mem_limit}")
        r2_best_df = imp.fetch(sql_r2_best)
else:
    r2_best_df = pd.DataFrame(columns=['r2_id', 'cft_id', 'dog_contract_number', 'join_type'])

if len(r2_best_df):
    r2_best_df['dog_contract_number'] = r2_best_df['dog_contract_number'].apply(normalize_contract)
    r2_best_df = r2_best_df.drop_duplicates('r2_id')

len(r2_best_df)

## 7) Apply fallback only where agr_id is null

In [ ]:
base_tier1 = agr_df[agr_df['agr_id'].notna()].copy()
base_tier1 = base_tier1.merge(
    r2_best_df[['r2_id', 'cft_id', 'dog_contract_number', 'join_type']],
    left_on='agr_id',
    right_on='r2_id',
    how='left'
)
base_tier1 = base_tier1.merge(inn_cdi_cft_df[['inn', 'cdi_id', 'cft_id']].drop_duplicates('inn'), on='inn', how='left', suffixes=('', '_from_cdi'))
base_tier1['cft_id'] = base_tier1['cft_id'].where(base_tier1['cft_id'].notna(), base_tier1.get('cft_id_from_cdi'))
if 'cft_id_from_cdi' in base_tier1.columns:
    base_tier1 = base_tier1.drop(columns=['cft_id_from_cdi'])
base_tier1['fallback_r2_id'] = None
base_tier1['fallback_join_type'] = None

base_tier2 = agr_df[agr_df['agr_id'].isna()].copy()
if len(base_tier2):
    t2 = base_tier2.merge(inn_cdi_cft_df[['inn', 'cdi_id', 'cft_id']], on='inn', how='left')
    cft_values = sorted(t2['cft_id'].dropna().unique().tolist())
    if len(cft_values):
        cft_values_sql = ', '.join(["'" + str(x).replace("'", "''") + "'" for x in cft_values])
        sql_r2_by_cft = f"""
        with m as (
          select
            cast(id as string) as r2_id,
            cast(c_cl_org as string) as cft_id,
            cast(c_code_in_pr as string) as c_code_in_pr
          from ods.scd1_z_r2_ip_merchants
          where cast(c_cl_org as string) in ({cft_values_sql})
        ),
        d as (
          select
            cast(id as string) as dog_id,
            cast(c_parent_id as string) as dog_parent_id,
            cast(c_code_in_pr as string) as dog_code_in_pr,
            cast(c_num as string) as dog_contract_number
          from ods.scd1_z_r2_ip_merch_dog
        ),
        candidate as (
          select m.r2_id, m.cft_id, d.dog_contract_number, 'join_by_parent_id' as join_type, 1 as rank, d.dog_id
          from m left join d on d.dog_parent_id = m.r2_id
          union all
          select m.r2_id, m.cft_id, d.dog_contract_number, 'join_by_id' as join_type, 2 as rank, d.dog_id
          from m left join d on d.dog_id = m.r2_id
          union all
          select m.r2_id, m.cft_id, d.dog_contract_number, 'join_by_code_in_pr' as join_type, 3 as rank, d.dog_id
          from m left join d on d.dog_code_in_pr is not null and m.c_code_in_pr is not null and d.dog_code_in_pr = m.c_code_in_pr
          union all
          select m.r2_id, m.cft_id, cast(null as string) as dog_contract_number, 'no_match' as join_type, 9 as rank, cast(null as string) as dog_id
          from m
        ),
        ranked as (
          select *, row_number() over (
            partition by r2_id
            order by rank, case when dog_contract_number is null then 1 else 0 end, dog_id
          ) as rn
          from candidate
        )
        select r2_id, cft_id, dog_contract_number, join_type
        from ranked
        where rn = 1
        """
        with imp:
            imp.execute(f"set MEM_LIMIT={mem_limit}")
            r2_cft_df = imp.fetch(sql_r2_by_cft)
        r2_cft_df['dog_contract_number'] = r2_cft_df['dog_contract_number'].apply(normalize_contract)
    else:
        r2_cft_df = pd.DataFrame(columns=['r2_id', 'cft_id', 'dog_contract_number', 'join_type'])

    t2 = t2.merge(r2_cft_df[['r2_id', 'cft_id', 'dog_contract_number', 'join_type']], on='cft_id', how='left')
    t2['match_priority'] = 3
    t2.loc[
        t2['agreement_contract_number'].notna() &
        t2['dog_contract_number'].notna() &
        (t2['agreement_contract_number'] == t2['dog_contract_number']),
        'match_priority'
    ] = 1
    t2 = t2.sort_values(['row_uid', 'match_priority', 'r2_id'])
    t2_best = t2.groupby('row_uid', as_index=False).first()

    base_tier2 = base_tier2.merge(
        t2_best[['row_uid', 'cdi_id', 'cft_id', 'dog_contract_number', 'r2_id', 'join_type']],
        on='row_uid',
        how='left'
    )
    base_tier2['fallback_r2_id'] = base_tier2['r2_id']
    base_tier2['fallback_join_type'] = base_tier2['join_type']
else:
    base_tier2['cdi_id'] = None
    base_tier2['cft_id'] = None
    base_tier2['dog_contract_number'] = None
    base_tier2['r2_id'] = None
    base_tier2['join_type'] = None
    base_tier2['fallback_r2_id'] = None
    base_tier2['fallback_join_type'] = None

base_df_raw = pd.concat([base_tier1, base_tier2], ignore_index=True, sort=False)
base_df_raw['dog_contract_number'] = base_df_raw['dog_contract_number'].apply(normalize_contract)
base_df_raw['contract_number_final'] = base_df_raw['dog_contract_number'].where(base_df_raw['dog_contract_number'].notna(), base_df_raw['agreement_contract_number'])
base_df_raw['agr_id_final'] = base_df_raw['agr_id'].where(base_df_raw['agr_id'].notna(), base_df_raw['fallback_r2_id'])
base_df_raw['key_source'] = base_df_raw['agr_id'].apply(lambda x: 'agr_id' if pd.notna(x) and str(x).strip() != '' else 'r2_fallback')
base_df_raw['join_type'] = base_df_raw['join_type'].fillna('no_match')

base_df_raw = base_df_raw[[
    'row_uid', 'inn', 'agr_id', 'r2_id', 'fallback_r2_id', 'key_source', 'join_type',
    'agreement_contract_number', 'dog_contract_number', 'contract_number_final',
    'cdi_id', 'cft_id', 'agr_id_final'
]].drop_duplicates().reset_index(drop=True)

len(base_df_raw)

## 8) Minimal final dataset for root-cause

In [ ]:
base_df_final = base_df_raw.copy()

# For root-cause checks we only need final key fields and chain identifiers
base_df_final = base_df_final[[
    "row_uid", "inn", "agr_id", "r2_id", "fallback_r2_id",
    "key_source", "join_type", "agreement_contract_number",
    "dog_contract_number", "contract_number_final", "cdi_id",
    "cft_id", "agr_id_final"
]].drop_duplicates().reset_index(drop=True)

len(base_df_final)

## 9) Root-cause prep (minimal)

In [ ]:
lake_inn_set = set(base_df_final['inn'].dropna().tolist())
only_excel_set = excel_inn_set - lake_inn_set
only_lake_set = lake_inn_set - excel_inn_set

pd.DataFrame([
    {'metric': 'excel_unique_inn', 'value': len(excel_inn_set)},
    {'metric': 'lake_unique_inn', 'value': len(lake_inn_set)},
    {'metric': 'inn_only_in_excel', 'value': len(only_excel_set)},
    {'metric': 'inn_only_in_lake', 'value': len(only_lake_set)},
])

## Root-cause checks for tails

Автоматический разбор:
- ИНН, которые есть только в Excel (`inn_only_in_excel`);
- строк, где не удалось получить `agr_id_final`.

In [ ]:
# 1) INN only in Excel: check whether they exist in source SA perimeter
only_excel_sorted = sorted(list(only_excel_set))

if len(only_excel_sorted):
    only_excel_probe_df = pd.DataFrame({'inn': only_excel_sorted})
    src_presence_df = only_excel_probe_df.merge(
        agr_df[['inn', 'agr_id', 'agreement_contract_number']].drop_duplicates(),
        on='inn',
        how='left'
    )
    src_presence_df['present_in_agr_source'] = src_presence_df['agr_id'].notna() | src_presence_df['agreement_contract_number'].notna()
    src_presence_stat_df = pd.DataFrame([
        {'metric': 'only_excel_inn_cnt', 'value': len(only_excel_probe_df)},
        {'metric': 'only_excel_present_in_agr_source_cnt', 'value': int(src_presence_df['present_in_agr_source'].sum())},
        {'metric': 'only_excel_not_in_agr_source_cnt', 'value': int((~src_presence_df['present_in_agr_source']).sum())},
    ])
else:
    src_presence_df = pd.DataFrame(columns=['inn', 'agr_id', 'agreement_contract_number', 'present_in_agr_source'])
    src_presence_stat_df = pd.DataFrame([
        {'metric': 'only_excel_inn_cnt', 'value': 0},
        {'metric': 'only_excel_present_in_agr_source_cnt', 'value': 0},
        {'metric': 'only_excel_not_in_agr_source_cnt', 'value': 0},
    ])

src_presence_stat_df, src_presence_df.head(50)

In [ ]:
# 2) Missing agr_id_final: locate first failed link in chain
missing_agr_id_final_df = base_df_final[base_df_final['agr_id_final'].isna()].copy() if 'base_df_final' in globals() else base_df[base_df['agr_id_final'].isna()].copy()

if len(missing_agr_id_final_df):
    rc_df = missing_agr_id_final_df[[
        'row_uid', 'inn', 'agr_id', 'agreement_contract_number', 'dog_contract_number',
        'contract_number_final', 'key_source', 'join_type', 'cdi_id', 'cft_id', 'r2_id', 'fallback_r2_id', 'agr_id_final'
    ]].drop_duplicates()

    rc_df['has_agr_id_src'] = rc_df['agr_id'].notna() & (rc_df['agr_id'].astype(str) != '')
    rc_df['has_cdi'] = rc_df['cdi_id'].notna() & (rc_df['cdi_id'].astype(str) != '')
    rc_df['has_cft'] = rc_df['cft_id'].notna() & (rc_df['cft_id'].astype(str) != '')
    rc_df['has_r2'] = rc_df['r2_id'].notna() & (rc_df['r2_id'].astype(str) != '')
    rc_df['is_join_no_match'] = rc_df['join_type'].fillna('') == 'no_match'

    rc_df['root_cause_stage'] = 'resolved'
    rc_df.loc[~rc_df['has_cdi'], 'root_cause_stage'] = 'missing_inn_to_cdi'
    rc_df.loc[rc_df['has_cdi'] & ~rc_df['has_cft'], 'root_cause_stage'] = 'missing_cdi_to_cft'
    rc_df.loc[rc_df['has_cft'] & ~rc_df['has_r2'], 'root_cause_stage'] = 'missing_cft_to_r2'
    rc_df.loc[rc_df['has_r2'] & rc_df['is_join_no_match'], 'root_cause_stage'] = 'r2_join_type_no_match'
    rc_df.loc[rc_df['has_agr_id_src'] & ~rc_df['has_r2'], 'root_cause_stage'] = 'agr_id_not_found_in_r2'

    root_cause_stat_df = rc_df.groupby('root_cause_stage', as_index=False).agg(
        rows=('row_uid', 'size'),
        inn_cnt=('inn', 'nunique')
    ).sort_values(['rows', 'inn_cnt'], ascending=False)

    root_cause_examples_df = rc_df.sort_values(['root_cause_stage', 'inn', 'row_uid']).head(100).reset_index(drop=True)
else:
    rc_df = pd.DataFrame(columns=[
        'row_uid', 'inn', 'agr_id', 'agreement_contract_number', 'dog_contract_number',
        'contract_number_final', 'key_source', 'join_type', 'cdi_id', 'cft_id', 'r2_id', 'fallback_r2_id', 'agr_id_final',
        'has_agr_id_src', 'has_cdi', 'has_cft', 'has_r2', 'is_join_no_match', 'root_cause_stage'
    ])
    root_cause_stat_df = pd.DataFrame(columns=['root_cause_stage', 'rows', 'inn_cnt'])
    root_cause_examples_df = pd.DataFrame(columns=rc_df.columns)

root_cause_stat_df, root_cause_examples_df

In [ ]:
# 3) Compact diagnostic output
base_for_rc = base_df_final if 'base_df_final' in globals() else base_df
root_cause_kpi_df = pd.DataFrame([
    {
        'metric': 'missing_agr_id_final_rows',
        'value': int(base_for_rc['agr_id_final'].isna().sum())
    },
    {
        'metric': 'missing_agr_id_final_inn',
        'value': int(base_for_rc.loc[base_for_rc['agr_id_final'].isna(), 'inn'].nunique())
    },
    {
        'metric': 'only_excel_inn_cnt',
        'value': len(only_excel_set)
    },
])

root_cause_kpi_df, src_presence_stat_df, root_cause_stat_df, root_cause_examples_df.head(30)